In [1]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv('Combined_Flights_Clean.csv')

In [2]:
n_avant = len(df)
n_cancelled = df["Cancelled"].sum()
n_diverted = df["Diverted"].sum()

df = df[~df["Cancelled"] & ~df["Diverted"]].copy()
df = df.drop(columns=["Cancelled", "Diverted"])

print(f"Lignes retirées (Cancelled) : {n_cancelled:,}")
print(f"Lignes retirées (Diverted) : {n_diverted:,}")
print(
    f"Shape après filtrage : {df.shape} ({(n_avant - len(df))/n_avant:.2%} retiré)"
)

Lignes retirées (Cancelled) : 153,629
Lignes retirées (Diverted) : 20,791
Shape après filtrage : (7917264, 19) (2.16% retiré)


In [3]:
masque_incoherent = df["CRSElapsedTime"] <= 0
n_incoherent = masque_incoherent.sum()

lignes_incoherentes = df[masque_incoherent].copy()
lignes_incoherentes.to_csv(
    "lignes_incoherentes_CRSElapsedTime.csv", index=False
)

df = df[~masque_incoherent].copy()

print(
    f"Lignes incohérentes (CRSElapsedTime <= 0) retirées : {n_incoherent:,} ({n_incoherent/n_avant:.4%})"
)

Lignes incohérentes (CRSElapsedTime <= 0) retirées : 4 (0.0000%)


In [4]:
manquants = df.isna().sum()
manquants = manquants[manquants > 0].sort_values(ascending=False)
if manquants.empty:
    print("\nAucune valeur manquante restante.")
else:
    print("\nValeurs manquantes restantes :")
    pct = (manquants / len(df) * 100).round(4)
    print(pd.DataFrame({"n_manquant": manquants, "%": pct}))

assert (
    df["ArrDel15"].isna().sum() == 0
), "ArrDel15 contient encore des valeurs manquantes"


Valeurs manquantes restantes :
                n_manquant    %
CRSElapsedTime           1  0.0


In [6]:
cols_categorielles = df.select_dtypes(include=["object", "string"]).columns.tolist()

mappings = {}
for col in cols_categorielles:
    if col == "FlightDate":
        continue
    categories = sorted(df[col].dropna().unique().tolist())
    mapping = {cat: i for i, cat in enumerate(categories)}
    mappings[col] = mapping
    df[col + "_encoded"] = df[col].map(mapping)

with open("encodage_categoriel_mappings.json", "w") as f:
    json.dump(mappings, f, ensure_ascii=False, indent=2)

In [7]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
df["is_weekend"] = df["DayOfWeek"].isin([6, 7]).astype(int)

print(f"\nShape final : {df.shape}")



Shape final : (7917260, 26)


In [8]:
df.to_csv("flights_prepared.csv", index=False)